In [4]:
import os
import PIL as pil
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
from torchsummary import summary
from torchvision.utils import make_grid
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import math

In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/"
BATCH_SIZE = 32 
NUM_WORKERS = os.cpu_count()-2
USE_AMP = True # Automatic Mixed Precision

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
def get_folder_structure(base="data"):
    """
    Returns the folder structure of the data folder
    """
    return [x[0] for x in os.walk(base)]


get_folder_structure()

In [ ]:
class BenGrahamPreprocessing:
    def __init__(self, img_size):
        self.img_size = img_size

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        image_np = np.array(image)

        # Aplicar el desenfoque gaussiano de OpenCV
        blurred_image = cv2.GaussianBlur(image_np, (0, 0), self.img_size / 10)

        # Combinar la imagen original y la desenfocada
        combined_image = cv2.addWeighted(image_np, 4, blurred_image, -4, 128)

        # Convertir a escala de grises
        return F.to_pil_image(combined_image)

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        #  transforms.Grayscale(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        BenGrahamPreprocessing(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225]),
    ]
)

In [ ]:
# Load the training data from the data folder
trainset = torchvision.datasets.ImageFolder(root="data/trainbin/", transform=transform)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
testset = torchvision.datasets.ImageFolder(root="data/testbin/", transform=transform)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)

In [ ]:
print("Follwing classes are there : \n", trainset.classes)

In [ ]:
def display_img(img, label):
    print(f"Label : {trainset.classes[label]}")
    img_valid_range = np.clip(img.numpy().transpose((1, 2, 0)), 0, 1)
    plt.imshow(img_valid_range)


# display the first image in the dataset
display_img(*trainset[0])

In [ ]:
num_classes = len(trainset.classes)
# Cargar un modelo preentrenado
model = torchvision.models.resnet152(weights=torchvision.models.ResNet152_Weights.IMAGENET1K_V2)

# Freeze all layers except the last ones
for param in model.parameters():
    param.requires_grad = False

# Modify the final layer for our classification task
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

# Move model to device
model = model.to(device)
# model = torch.compile(model)

In [ ]:
summary(model, (3, IMG_SIZE, IMG_SIZE), device=device.type)

In [ ]:
criterion = torch.nn.NLLLoss(reduction="sum")  #  Use NLLLoss for binary-class classification
# criterion = torch.nn.CrossEntropyLoss()  # Use CrossEntropy for multi-class classification
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-4)  # Only optimize parameters of final layer
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

In [ ]:
torch.set_num_threads(NUM_WORKERS)  # Usa todos los cores
torch.backends.cudnn.benchmark = True  # Mejora la velocidad de entrenamiento en GPU

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # loss.backward()
            # optimizer.step()
            for param in model.parameters():
                param.grad = None
            # optimizer.zero_grad()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(trainloader.dataset)
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in tqdm(testloader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = correct / total
        print(f'Validation Accuracy: {val_accuracy:.4f}')

In [ ]:
# def train_model(model, criterion, optimizer, num_epochs=10):
#     """
#     Entrena el modelo.

#     Args:
#         model: El modelo a entrenar.
#         criterion: La función de pérdida.
#         optimizer: El optimizador.
#         num_epochs: El número de épocas de entrenamiento.

#     Returns:
#         El modelo entrenado.
#     """
#     for epoch in range(num_epochs):
#         print(f"Epoch {epoch}/{num_epochs - 1}")
#         print("-" * 10)

#         # Iterar sobre las fases de entrenamiento y validación
#         for phase in ["train"]:  # , 'val']:
#             if phase == "train":
#                 model.train()  # Poner el modelo en modo de entrenamiento
#             else:
#                 model.eval()  # Poner el modelo en modo de evaluación

#             running_loss = 0.0
#             running_corrects = 0

#             # Iterar sobre los datos en batches
#             for inputs, labels in tqdm(trainloader):  # dataloaders[phase]:
#                 inputs = inputs.to(device)  # Mover los datos al dispositivo
#                 labels = labels.to(device)

#                 # Zero the parameter gradients
#                 optimizer.zero_grad()

#                 # Habilitar el cálculo de gradientes solo en la fase de entrenamiento
#                 with torch.set_grad_enabled(phase == "train"):
#                     # Forward pass: calcular las predicciones
#                     outputs = model(inputs)
#                     _, preds = torch.max(outputs, 1)  # Obtener la clase predicha
#                     loss = criterion(outputs, labels)  # Calcular la pérdida

#                     # Backward pass + optimizar solo en la fase de entrenamiento
#                     if phase == "train":
#                         loss.backward()  # Calcular los gradientes
#                         optimizer.step()  # Actualizar los pesos del modelo

#                 # Calcular estadísticas
#                 running_loss += loss.item() * inputs.size(0)
#                 running_corrects += torch.sum(preds == labels.data)

#             # Calcular la pérdida y la precisión para la época
#             epoch_loss = running_loss / len(trainloader.dataset)  # dataset_sizes[phase]
#             epoch_acc = running_corrects.double() / len(trainloader.dataset)  # dataset_sizes[phase]

#             print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

#         print()

#     print("Entrenamiento completado")
#     return model

In [ ]:
train_model(model, criterion, optimizer, num_epochs=10)

In [ ]:
model.eval()  # Establece modo de evaluación
correct = 0
total = 0

with torch.no_grad():  # Desactiva el cálculo de gradientes para evaluación
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Precisión en el conjunto de prueba: {100 * correct / total:.2f}%")

**Eliminar pixeles negras**

*VER1*

In [2]:
def recortar_bordes_negros(imagen):
    pixeles = imagen.load()
    ancho, alto = imagen.size
    ancho_medio= ancho//2
    alto_medio = alto//2

    x_inicio = 0
    for x in range(ancho):
        if pixeles[x, alto_medio] != (0, 0, 0):
            x_inicio = x
            break

    x_fin = ancho - 1
    for x in range(ancho - 1, -1, -1):
        if pixeles[x, alto_medio] != (0, 0, 0): 
            x_fin = x
            break
    
    y_inicio = 0
    for y in range(alto):
        if pixeles[ancho_medio, y] != (0, 0, 0): 
            y_inicio = y
            break

    y_fin = alto - 1
    for y in range(alto - 1, -1, -1):
        if pixeles[ancho_medio, y] != (0, 0, 0):
            y_fin = y
            break
    
    imagen_recortada = imagen.crop((x_inicio, y_inicio, x_fin + 1, y_fin + 1))
    return imagen_recortada



In [3]:
source_folder = "data/train"
destination_folder = "data/train_limpia"

os.makedirs(destination_folder, exist_ok=True)

subcarpetas = [f for f in os.listdir(source_folder) if os.path.isdir(os.path.join(source_folder, f))]

for subcarpeta in subcarpetas:
    destino_subcarpeta = os.path.join(destination_folder, subcarpeta)
    os.makedirs(destino_subcarpeta, exist_ok=True)
    
    source_subcarpeta = os.path.join(source_folder, subcarpeta)
    files = os.listdir(source_subcarpeta)
    
    image_files = [f for f in files if os.path.isfile(os.path.join(source_subcarpeta, f)) and f.endswith(('.jpg', '.png', '.jpeg'))]
    
    for file in image_files[:10]:
        source_path = os.path.join(source_subcarpeta, file)
        image = Image.open(source_path)
        image_recortada = recortar_bordes_negros(image)
        destination_path = os.path.join(destino_subcarpeta, file)
        image_recortada.save(destination_path)
    
    print(f"Imágenes de la carpeta '{subcarpeta}' procesadas y copiadas a '{destino_subcarpeta}'.")


Imágenes de la carpeta '0-noDR' procesadas y copiadas a 'data/train_limpia\0-noDR'.
Imágenes de la carpeta '1-mild' procesadas y copiadas a 'data/train_limpia\1-mild'.
Imágenes de la carpeta '2-moderate' procesadas y copiadas a 'data/train_limpia\2-moderate'.
Imágenes de la carpeta '3-severe' procesadas y copiadas a 'data/train_limpia\3-severe'.
Imágenes de la carpeta '4-proliferativeDR' procesadas y copiadas a 'data/train_limpia\4-proliferativeDR'.


*VER2*

In [ ]:
def encontrar_limites(imagen):
    pixeles = imagen.load()
    ancho, alto = imagen.size
    ancho_medio = ancho//2
    alto_medio = alto//2
    x_inicio = 0
    x_inicio = 0
    for x in range(ancho):
        if pixeles[x, alto_medio] != (0, 0, 0):
            x_inicio = x
            break

    x_fin = ancho - 1
    for x in range(ancho - 1, -1, -1):
        if pixeles[x, alto_medio] != (0, 0, 0): 
            x_fin = x
            break
    
    y_inicio = 0
    for y in range(alto):
        if pixeles[ancho_medio, y] != (0, 0, 0): 
            y_inicio = y
            break

    y_fin = alto - 1
    for y in range(alto - 1, -1, -1):
        if pixeles[ancho_medio, y] != (0, 0, 0):
            y_fin = y
            break
    
    return x_inicio, x_fin, y_inicio, y_fin

In [15]:
def calcular_centro_y_radio(x_inicio, x_fin, y_inicio, y_fin):
    centro_x = (x_inicio + x_fin) // 2
    centro_y = (y_inicio + y_fin) // 2
    radio = max((x_fin - x_inicio) // 2, (y_fin - y_inicio) // 2)
    return centro_x, centro_y, radio

In [16]:
def recortar_ojo(imagen, centro_x, centro_y, radio):
    lado = int((2 * radio) / math.sqrt(2))
    mitad_lado = lado // 2
    
    izquierda = max(centro_x - mitad_lado, 0)
    arriba = max(centro_y - mitad_lado, 0)
    derecha = min(centro_x + mitad_lado, imagen.width)
    abajo = min(centro_y + mitad_lado, imagen.height)
    
    imagen_recortada = imagen.crop((izquierda, arriba, derecha, abajo))
    return imagen_recortada

In [17]:
source_folder = "data/train"
destination_folder = "data/train_limpia2"

os.makedirs(destination_folder, exist_ok=True)

subcarpetas = [f for f in os.listdir(source_folder) if os.path.isdir(os.path.join(source_folder, f))]

for subcarpeta in subcarpetas:
    destino_subcarpeta = os.path.join(destination_folder, subcarpeta)
    os.makedirs(destino_subcarpeta, exist_ok=True)
    
    source_subcarpeta = os.path.join(source_folder, subcarpeta)
    files = os.listdir(source_subcarpeta)
    
    image_files = [f for f in files if os.path.isfile(os.path.join(source_subcarpeta, f)) and f.endswith(('.jpg', '.png', '.jpeg'))]
    
    for file in image_files[:10]:
        source_path = os.path.join(source_subcarpeta, file)
        image = Image.open(source_path)
        x_inicio, x_fin, y_inicio, y_fin = encontrar_limites(image)
        
        centro_x, centro_y, radio = calcular_centro_y_radio(x_inicio, x_fin, y_inicio, y_fin)
        
        imagen_recortada = recortar_ojo(image, centro_x, centro_y, radio)

        destination_path = os.path.join(destino_subcarpeta, file)
        imagen_recortada.save(destination_path)
    
    print(f"Imágenes de la carpeta '{subcarpeta}' procesadas y copiadas a '{destino_subcarpeta}'.")

print("Proceso completado.")

Imágenes de la carpeta '0-noDR' procesadas y copiadas a 'data/train_limpia2\0-noDR'.
Imágenes de la carpeta '1-mild' procesadas y copiadas a 'data/train_limpia2\1-mild'.
Imágenes de la carpeta '2-moderate' procesadas y copiadas a 'data/train_limpia2\2-moderate'.
Imágenes de la carpeta '3-severe' procesadas y copiadas a 'data/train_limpia2\3-severe'.
Imágenes de la carpeta '4-proliferativeDR' procesadas y copiadas a 'data/train_limpia2\4-proliferativeDR'.
Proceso completado.
